In [2]:
# script para generar los archivos de montos, sucursales y ventas a partir del archivo de ventas original
# como usar => primero descargar el file de tibco, cambiar el nombre a "Ventas_Rango_de_fechas.xls" 
# moverlo a la carpeta ventas_mod, luego abrir el archivo en excel y eliminar la primera fila y cambiar la columna monto a enteros, 
# sin decimales, luego ejecutar este script. Se generarán los archivos montos.csv, sucursales.csv y ventas.csv en la carpeta ventas_mod


import pandas as pd
import xlrd

df = pd.read_excel('ventas_mod/Ventas_Rango_de_fechas.xls')
pd.set_option('display.width', None)
pd.set_option('display.max_columns', None)
pd.set_option('display.max_colwidth', None)
pd.set_option('display.max_rows', None)

for col in ['FAMILIA', 'PRODUCTO', 'VARIACION']:
    df[col] = df[col].astype(str).str.strip().str.replace(r'\s+', ' ', regex=True)

montos = df.copy()
sucursales = df.copy()
ventas = df.copy()

# tabla de montos
montos = montos.drop(columns=['SUCURSAL', 'CANTIDAD']).groupby(['FAMILIA', 'PRODUCTO', 'VARIACION'], dropna=False)['MONTO'].sum().reset_index()
#display(montos)


#tabla de sucursales
columnas_table_sucursales = [col for col in df.columns if col not in ['SUCURSAL', 'MONTO']]
sucursales = sucursales.drop(columns=columnas_table_sucursales).groupby('SUCURSAL', dropna=False).sum().reset_index()
#display(sucursales)



## tabla de ventas por sucursal
columnas_index = [col for col in ventas.columns if col not in ['SUCURSAL', 'CANTIDAD']]
for col in columnas_index:
    ventas[col] = ventas[col].fillna('SIN DATO')

ventas = ventas.pivot_table(index=columnas_index, columns='SUCURSAL', values='CANTIDAD', aggfunc='sum', fill_value=0).reset_index()
ventas = ventas.drop(columns=['MONTO'])

for col in ['FAMILIA', 'PRODUCTO', 'VARIACION']:
    ventas[col] = ventas[col].astype(str).str.strip().str.replace(r'\s+', ' ', regex=True)

columnas_ventas = [col for col in ventas.columns if col not in ['FAMILIA', 'PRODUCTO', 'VARIACION']]
columnas_identificadoras = ['FAMILIA', 'PRODUCTO', 'VARIACION']
ventas = ventas.groupby(columnas_identificadoras, dropna=False)[columnas_ventas].sum().reset_index()

ventas.columns.name = None   # <- elimina el "SUCURSAL" fantasma en el índice
ventas[['FAMILIA', 'PRODUCTO', 'VARIACION']] = ventas[['FAMILIA', 'PRODUCTO', 'VARIACION']].replace('SIN DATO', pd.NA)


ventas = ventas.rename(columns={'C.C. TITAN': 'TITAN', 'CC PARQUE COLINA': 'COLINA'})

ventas = ventas.merge(montos, on=['FAMILIA', 'PRODUCTO', 'VARIACION'], how='left')
ventas = ventas.rename(columns={'MONTO': 'VALOR'})
ventas['VALOR'] = ventas['VALOR'].astype('Int64')
display(ventas)
montos.to_csv('ventas_mod/montos.csv', sep='|', index=False)
sucursales.to_csv('ventas_mod/sucursales.csv', sep='|', index=False)
ventas.to_csv('ventas_mod/ventas.csv', sep='|', index=False)





,FAMILIA,PRODUCTO,VARIACION,ANDINO,TITAN,CALLE 109,CALLE 122,CALLE 79,COLINA,CEDRITOS,CHIA,FABRICA,FONTANAR,SALITRE,SANTAFE,VALOR
0,ARITOS,ARITO,NaN,13,9,11,33,12,25,29,16,33,19,16,16,5685300
1,ARITOS,ARITO AREQUIPE,NaN,6,5,0,2,2,3,3,2,0,2,2,3,721800
2,ARITOS,ARITO CHOCOFLAN,NaN,2,2,1,1,1,0,2,2,0,0,2,0,310700
3,ARITOS,ARITO CHOCOLATE,NaN,4,4,2,1,3,4,4,5,0,2,2,1,764800
4,ARITOS,ARITO LIMON,NaN,13,8,3,9,7,10,10,6,5,2,6,9,2136840
5,ARITOS,ARITO MARACUYA,NaN,7,8,3,7,2,8,10,6,2,5,7,7,1728120
6,ARITOS,ARITO PISTACHO,NaN,0,0,0,0,0,1,0,0,0,0,0,0,23900
7,BEBIDAS CALIENTES,AMERICANO,NaN,48,33,13,36,13,61,20,35,0,33,36,24,2851200
8,BEBIDAS CALIENTES,AMERICANO DOBLE,NaN,0,1,0,1,0,1,0,0,0,0,0,0,26100
9,BEBIDAS CALIENTES,AROMATICA HIERBABUENA,NaN,3,6,0,5,1,1,0,1,0,5,6,2,207000


In [4]:
# script para generar el archivo resultado.csv a partir de los archivos ventas.csv y test_stock.txt
# este debe ser nombre de las columnas del archivo stock.txt debe ser el mismo que el de las columnas del archivo ventas.csv, 
# excepto por la columna valor que solo está en ventas.csv

import pandas as pd

columnas_file_ventas = [
    'familia',
    'producto',
    'variacion',
    'salitre',
    'andino',
    'titan',
    'colina',
    'santafe',
    'fontanar',
    'chia',
    'calle 79',
    'calle 109',
    'calle 122',
    'cedritos',
    'fabrica',
    'valor'
]

columnas_file_stock = [
    'familia',
    'producto',
    'variacion',
    'salitre',
    'andino',
    'titan',
    'colina',
    'santafe',
    'fontanar',
    'chia',
    'calle 79',
    'calle 109',
    'calle 122',
    'cedritos',
]


sucursales = [
    "salitre",
    "andino",
    "titan",
    "colina",
    "santafe",
    "fontanar",
    "chia",
    "calle 79",
    "calle 109",
    "calle 122",
    "cedritos"
]


stock = pd.read_csv('test_stock.txt', sep='|')
ventas = pd.read_csv('ventas_mod/ventas.csv', sep='|')



ventas.columns = ventas.columns.str.lower()
stock.columns = stock.columns.str.lower()



if set(ventas.columns.tolist()) != set(columnas_file_ventas):
    raise ValueError("Las columnas del archivo de ventas no coinciden con las esperadas")

if set(stock.columns.tolist()) != set(columnas_file_stock):
    raise ValueError("Las columnas del archivo de stock no coinciden con las esperadas")



stock = stock.fillna('.')
ventas = ventas.fillna('.')
pd.set_option('display.width', None)
pd.set_option('display.max_columns', None)
pd.set_option('display.max_colwidth', None)
pd.set_option('display.max_rows', None)



ventas['productos'] = ventas['producto'].str.strip()
ventas['variacion'] = ventas['variacion'].str.strip()
ventas['nombre'] = ventas['productos'].str.upper() + ' ' + ventas['variacion'].str.upper()
ventas['nombre'] = ventas['familia'].str.upper() + ' ' + ventas['producto'].str.upper() + ' ' + ventas['variacion'].str.upper()

stock['producto'] = stock['producto'].str.strip()
stock['variacion'] = stock['variacion'].str.strip()
stock['nombre'] = stock['familia'].str.upper() + ' ' + stock['producto'].str.upper() + ' ' + stock['variacion'].str.upper()


stock['nombre'] = stock['nombre'].astype(str).str.strip().str.replace(r'\s+', ' ', regex=True)
ventas['nombre'] = ventas['nombre'].astype(str).str.strip().str.replace(r'\s+', ' ', regex=True)


stock = stock.rename(
    columns={col: f"{col}_stock" for col in sucursales}
)
resultado = pd.merge(
    ventas,
    stock,
    on="nombre",
    how="outer",
    suffixes=("_ventas", "_stock")
)

#array de columnas para el resultado final
columnas = []
#columnas.append('familia_ventas')
columnas.append('nombre')
for sucursal in sucursales:
    
    columnas.append(f"{sucursal}")
    columnas.append(f"{sucursal}_stock")

columnas.append('fabrica')
columnas.append('valor')




for col in resultado.columns:
    if pd.api.types.is_numeric_dtype(resultado[col]):
        resultado[col] = pd.to_numeric(
            resultado[col], errors='coerce'
        ).round().astype('Int64')

mask = resultado['familia_ventas'].isna()

cols_numericas = resultado.select_dtypes(include='number').columns

resultado.loc[mask, cols_numericas] = (
    resultado.loc[mask, cols_numericas].fillna(0)
)


# poner valores vacios en las columnas stock
sucursales_stock = [f"{i}_stock" for i in sucursales]

resultado[sucursales_stock] = resultado[sucursales_stock].astype("Int64")


mask = ((resultado['familia_ventas'].isin([
   'BEBIDAS CALIENTES',
])) | (resultado['nombre'].isin([
    'MINI BITES POR 12 UNIDADES .',
    'MINI BITES POR 4 UNIDADES .',
    'MINI BITES POR 8 UNIDADES .'])))

resultado.loc[mask, sucursales_stock] = pd.NA
resultado['valor'] = pd.to_numeric(resultado['valor'], errors='coerce').round().astype('Int64')

resultado[columnas].to_csv('resultado.csv', sep='|', index=False)
#display(resultado[resultado['familia_ventas'] == 'BEBIDAS CALIENTES'][columnas])
#display(resultado[resultado['nombre'] == 'MINI BITES POR 12 UNIDADES .'][columnas])
display(resultado[columnas])
#display(ventas[ventas['producto'].str.contains('TARTALETA', na=False)])
#display(stock[stock['familia'].str.contains('BEBIDAS CALIENTES', na=False)])

,nombre,salitre,salitre_stock,andino,andino_stock,titan,titan_stock,colina,colina_stock,santafe,santafe_stock,fontanar,fontanar_stock,chia,chia_stock,calle 79,calle 79_stock,calle 109,calle 109_stock,calle 122,calle 122_stock,cedritos,cedritos_stock,fabrica,valor
0,ARITOS ARITO .,16,9,13,11,9,13,25,0,16,3,19,1,16,8,12,3,11,6,33,-7,29,4,33,5685300
1,ARITOS ARITO AREQUIPE .,2,1,6,0,5,0,3,0,3,0,2,0,2,1,2,1,0,2,2,2,3,1,0,721800
2,ARITOS ARITO CHOCOFLAN .,2,0,2,1,2,0,0,2,0,0,0,1,2,0,1,1,1,0,1,0,2,0,0,310700
3,ARITOS ARITO CHOCOLATE .,2,3,4,1,4,0,4,0,1,2,2,1,5,0,3,2,2,1,1,4,4,2,0,764800
4,ARITOS ARITO LIMON .,6,6,13,5,8,4,10,1,9,0,2,4,6,2,7,2,3,4,9,2,10,3,5,2136840
5,ARITOS ARITO MARACUYA .,7,4,7,1,8,0,8,1,7,0,5,0,6,0,2,2,3,3,7,1,10,0,2,1728120
6,ARITOS ARITO PISTACHO .,0,0,0,0,0,0,1,-1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,23900
7,BEBIDAS CALIENTES AMERICANO .,36,<NA>,48,<NA>,33,<NA>,61,<NA>,24,<NA>,33,<NA>,35,<NA>,13,<NA>,13,<NA>,36,<NA>,20,<NA>,0,2851200
8,BEBIDAS CALIENTES AMERICANO DOBLE .,0,<NA>,0,<NA>,1,<NA>,1,<NA>,0,<NA>,0,<NA>,0,<NA>,0,<NA>,0,<NA>,1,<NA>,0,<NA>,0,26100
9,BEBIDAS CALIENTES AROMATICA DE FRUTAS .,2,<NA>,3,<NA>,3,<NA>,2,<NA>,1,<NA>,5,<NA>,6,<NA>,1,<NA>,3,<NA>,4,<NA>,0,<NA>,0,273000
